# Ejercicio 01: Agregar Experiment Tracking con MLflow

## Objetivo

En este notebook tienes codigo que entrena un modelo `RandomForestRegressor` para predecir
la duracion de viajes de taxi en NYC. El modelo **ya funciona**, pero no tiene ningun
sistema de tracking.

Tu tarea es agregar **MLflow tracking** en las celdas marcadas con `# TODO`.

Consulta el archivo `exercises/README.md` para instrucciones detalladas de cada paso.

---

## Prerequisitos

1. Haber ejecutado `notebooks/00_data_preparation.ipynb`
2. Tener el servidor MLflow corriendo en `http://127.0.0.1:5000`

---
## Parte 1: Codigo base (NO modificar, solo ejecutar)

Las siguientes celdas cargan los datos, entrenan el modelo y calculan metricas.
Ejecutalas todas antes de continuar.

In [ ]:
import os
import pickle

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

In [ ]:
# --- Cargar datos preprocesados ---
DATA_PATH = os.path.join("..", "data", "processed")

def load_pickle(filepath):
    with open(filepath, "rb") as f:
        return pickle.load(f)

X_train = load_pickle(os.path.join(DATA_PATH, "X_train.pkl"))
y_train = load_pickle(os.path.join(DATA_PATH, "y_train.pkl"))
X_val = load_pickle(os.path.join(DATA_PATH, "X_val.pkl"))
y_val = load_pickle(os.path.join(DATA_PATH, "y_val.pkl"))

print(f"Train: {X_train.shape[0]} filas, {X_train.shape[1]} features")
print(f"Val:   {X_val.shape[0]} filas, {X_val.shape[1]} features")

In [ ]:
# --- Entrenar modelo ---
n_estimators = 100
max_depth = 10
random_state = 0

rf = RandomForestRegressor(
    n_estimators=n_estimators,
    max_depth=max_depth,
    random_state=random_state,
)
rf.fit(X_train, y_train)

y_pred = rf.predict(X_val)
print("Modelo entrenado.")

In [ ]:
# --- Calcular metricas ---
rmse = np.sqrt(mean_squared_error(y_val, y_pred))
mae = mean_absolute_error(y_val, y_pred)
r2 = r2_score(y_val, y_pred)

print(f"RMSE: {rmse:.4f}")
print(f"MAE:  {mae:.4f}")
print(f"R2:   {r2:.4f}")

In [ ]:
# --- Generar tabla de predicciones ---
df_predictions = pd.DataFrame({
    "y_true": y_val,
    "y_pred": y_pred,
    "error": y_val - y_pred,
})
df_predictions.to_csv("predictions.csv", index=False)
print(f"Predicciones guardadas en predictions.csv ({len(df_predictions)} filas)")
df_predictions.head()

In [ ]:
# --- Generar grafica de residuales ---
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(y_val - y_pred, bins=50, edgecolor="black", alpha=0.7)
ax.set_title("Distribucion de residuales (y_true - y_pred)")
ax.set_xlabel("Residual (minutos)")
ax.set_ylabel("Frecuencia")
ax.axvline(x=0, color="red", linestyle="--", alpha=0.5)
fig.tight_layout()
fig.savefig("residuals.png", dpi=150)
plt.show()
print("Grafica guardada en residuals.png")

---

## Hasta aqui todo funciona, pero...

Los resultados solo estan en la consola y en archivos sueltos. Si mañana quieres
comparar este entrenamiento con otro, no tienes forma organizada de hacerlo.

**Tu tarea:** agregar MLflow tracking en las celdas siguientes para registrar todo
de forma estructurada.

---

## Parte 2: Agregar MLflow Tracking (completar los TODOs)

### TODO 1: Configurar la conexion a MLflow

Importa `mlflow`, conectate al servidor y selecciona un experimento.

**Pistas:**
- Usa `mlflow.set_tracking_uri(...)` con la URL del servidor
- Usa `mlflow.set_experiment(...)` con un nombre descriptivo

In [ ]:
# TODO 1: Configurar conexion a MLflow
# Escribe tu codigo aqui:



### TODO 2-5: Registrar el experimento completo

En la siguiente celda, usa `with mlflow.start_run():` y dentro registra:

- **TODO 2:** Tags con `mlflow.set_tag(clave, valor)`
  - `"problem_type"` -> `"regression"`
  - `"model_family"` -> `"random_forest"`
  - `"dataset"` -> `"nyc_green_taxi_2023_01_02"`


- **TODO 3:** Parametros con `mlflow.log_param(nombre, valor)`
  - Los 3 hiperparametros: `n_estimators`, `max_depth`, `random_state`


- **TODO 4:** Metricas con `mlflow.log_metric(nombre, valor)`
  - Las 3 metricas: `rmse`, `mae`, `r2`


- **TODO 5:** Artefactos con `mlflow.log_artifact(ruta_archivo)`
  - El CSV: `"predictions.csv"`
  - La grafica: `"residuals.png"`

In [ ]:
# TODO 2-5: Registrar el experimento completo en MLflow
# Escribe tu codigo aqui:

with mlflow.start_run(run_name="rf_ejercicio_01"):

    # TODO 2: Registrar tags


    # TODO 3: Registrar parametros


    # TODO 4: Registrar metricas


    # TODO 5: Registrar artefactos


    print("Run registrado en MLflow!")

### Verificacion

Ejecuta la siguiente celda para confirmar que tu run se registro correctamente.
Deberia mostrar una tabla con tus metricas.

In [ ]:
# --- Verificar que el run quedo registrado ---
runs = mlflow.search_runs(
    experiment_names=["nyc-taxi-ejercicio-01"],
    order_by=["metrics.rmse ASC"],
)

if len(runs) == 0:
    print("No se encontraron runs. Revisa que completaste los TODOs correctamente.")
    print("Verifica tambien que el nombre del experimento sea 'nyc-taxi-ejercicio-01'.")
else:
    cols = ["run_id", "metrics.rmse", "metrics.mae", "metrics.r2", "tags.model_family"]
    available_cols = [c for c in cols if c in runs.columns]
    print(f"Se encontraron {len(runs)} run(s):")
    display(runs[available_cols])

---

## Paso final: Revisar en la MLflow UI

1. Abre `http://127.0.0.1:5000` en tu navegador
2. Busca el experimento **nyc-taxi-ejercicio-01** en la barra lateral
3. Haz clic en tu run y verifica que aparezcan los 3 tags, 3 parametros, 3 metricas y 2 artefactos

Si todo aparece: **ejercicio completado!**

---

## Bonus (opcional)

1. Cambia `max_depth` a `15` y `n_estimators` a `200`, vuelve a ejecutar todo el notebook.
   Ahora deberias tener 2 runs. Compara las metricas en la UI.

2. Genera una grafica de scatter `y_true vs y_pred` y registrala como artefacto adicional.

3. Reemplaza todo el logging manual por `mlflow.autolog()` y compara que registra
   automaticamente vs lo que registraste a mano.